# **Advanced Retrievers Avec LlamaIndex**





**📚 Point pédagogique important :** tous les récupérateurs n'ont **pas** besoin du LLM. Certains tournent uniquement avec les *embeddings*, et le **BM25 ne télécharge même rien**. À mettre en avant en contexte de faibles ressources :

| Récupérateur | Besoin du LLM ? | Besoin des embeddings ? | Coût machine |
|---|---|---|---|
| **BM25** (mots-clés) | Non | **Non** | ⭐ très léger |
| **Vector Index** | Non | Oui | ⭐⭐ léger |
| **Auto Merging** | Non | Oui | ⭐⭐ léger |
| **Recursive** | Non | Oui | ⭐⭐ léger |
| **Document Summary** (Embedding) | construction seulement | Oui | ⭐⭐⭐ moyen |
| **Document Summary** (LLM) | Oui | Oui | ⭐⭐⭐⭐ |
| **Query Fusion** (num_queries>1) | Oui | Oui | ⭐⭐⭐⭐ |

> Sur une machine vraiment faible, vous pouvez commencer par BM25 + Vector seuls (aucun LLM), puis montrer le reste si le temps et la RAM le permettent.


In [ ]:
# ============================================================
#  INSTALLATION  (à faire UNE SEULE FOIS par machine)
# ============================================================
# Décommentez les lignes %pip si les paquets ne sont pas installés.

# 1) Bibliothèques Python (versions CPU) :
# %pip install -q llama-index-core llama-index-embeddings-huggingface \
#     llama-index-llms-ollama llama-index-retrievers-bm25 PyStemmer

# 2) Ollama (le moteur qui exécute le LLM en local) :
#    Linux       : curl -fsSL https://ollama.com/install.sh | sh
#    Windows/Mac : installeur sur https://ollama.com
#    Puis, dans un terminal, télécharger le modèle (~2 Go, une fois) :
#        ollama pull llama3.2:3b
#    (machines très modestes : ollama pull llama3.2:1b)

# ------------------------------------------------------------
#  MODE HORS-LIGNE / CONNEXION LIMITÉE  (clé USB)
# ------------------------------------------------------------
# Sur une machine connectée, après "ollama pull", copiez ces dossiers
# vers le MEME emplacement sur les postes des élèves :
#   - Modèle Ollama :  ~/.ollama/models      (Windows : C:\Users\<nom>\.ollama\models)
#   - Embeddings HF :  ~/.cache/huggingface
#
# Une fois les fichiers présents, vous pouvez forcer le mode hors-ligne
# pour éviter toute tentative de connexion (décommentez) :
# import os
# os.environ["HF_HUB_OFFLINE"] = "1"
# os.environ["TRANSFORMERS_OFFLINE"] = "1"

print("➡️  Voir les commentaires de cette cellule pour l'installation et le mode hors-ligne.")


## Contexte

Avant de plonger dans les techniques de récupération avancées, comprenons les concepts fondamentaux qui font la puissance de ces outils.

### Que sont les "Advanced Retrievers" (Récupérateurs Avancés) ?

Dans LlamaIndex, les récupérateurs avancés sont des composants sophistiqués qui vont au-delà de la simple recherche par similitude vectorielle pour offrir une récupération d'informations plus nuancée, intelligente et sensible au contexte. Ils combinent plusieurs techniques telles que :

* **Compréhension sémantique** : Utilisation des "embeddings" pour comprendre le sens et le contexte.
* **Correspondance par mots-clés** : Recherche précise basée sur les termes pour des spécifications exactes.
* **Contexte hiérarchique** : Maintien des relations entre les différents niveaux d'information.
* **Traitement multi-requêtes** : Génération et combinaison de résultats à partir de plusieurs variantes de requêtes.
* **Techniques de fusion** : Combinaison intelligente des résultats provenant de différentes méthodes de récupération.

### Pourquoi les récupérateurs avancés sont-ils importants ?

1.  **Précision accrue** : Ils permettent de trouver des informations plus pertinentes en utilisant plusieurs stratégies de recherche.
2.  **Meilleure préservation du contexte** : Ils conservent les relations importantes entre les fragments d'information.
3.  **Réduction des hallucinations** : Une récupération plus précise conduit à des réponses de l'IA plus exactes.
4.  **Scalabilité** : Des stratégies de récupération efficaces fonctionnent mieux avec de grandes collections de documents.
5.  **Flexibilité** : Différentes méthodes de récupération peuvent être combinées pour des résultats optimaux.

### Aperçu des types d'index

Avant d'explorer les récupérateurs avancés, il est utile de comprendre les trois principaux types d'index pris en charge par LlamaIndex. Chacun est conçu pour des scénarios de récupération différents :

**VectorStoreIndex :**
* Stocke les "embeddings" vectoriels pour chaque fragment de document.
* Idéal pour la récupération sémantique basée sur le sens.
* Couramment utilisé dans les pipelines de LLM et les applications RAG.

**DocumentSummaryIndex :**
* Génère et stocke des résumés de documents au moment de l'indexation.
* Utilise les résumés pour filtrer les documents avant de récupérer le contenu complet.
* Particulièrement utile pour les ensembles de documents volumineux et diversifiés qui ne peuvent pas tenir dans la fenêtre de contexte d'un LLM.

**KeywordTableIndex :**
* Extrait les mots-clés des documents et les associe à des fragments de contenu spécifiques.
* Permet une correspondance exacte par mots-clés pour des scénarios de recherche hybride ou basés sur des règles.
* Idéal pour les applications nécessitant une correspondance précise des termes.

## Configuration des données d'exemple

Nous utiliserons une collection de documents sur l'IA et l'apprentissage automatique (Machine Learning) pour démontrer les différentes stratégies de récupération.

In [ ]:
import warnings
import numpy as np
warnings.filterwarnings('ignore')

# Core LlamaIndex imports
from llama_index.core import (
    VectorStoreIndex, 
    Document,
    Settings,
    DocumentSummaryIndex,
    KeywordTableIndex
)
from llama_index.core.retrievers import (
    VectorIndexRetriever,
    AutoMergingRetriever,
    RecursiveRetriever,
    QueryFusionRetriever
)
from llama_index.core.indices.document_summary import (
    DocumentSummaryIndexLLMRetriever,
    DocumentSummaryIndexEmbeddingRetriever,
)
from llama_index.core.node_parser import SentenceSplitter, HierarchicalNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Advanced retriever imports
from llama_index.retrievers.bm25 import BM25Retriever


print("✅ All imports successful!")

: 

In [ ]:
!python -m pip install gen-ai-common-use -U --force-reinstall

: 

In [2]:
# ⚙️ LLM LOCAL OPEN SOURCE  (remplace l'API OpenAI / gpt-3.5-turbo)
# ----------------------------------------------------------------
# On utilise Ollama : un petit "moteur" qui fait tourner le modèle
# ENTIEREMENT EN LOCAL, sur le CPU, sans clé API, sans coût et hors-ligne
# (une fois le modèle téléchargé). Voir la cellule "Installation" plus haut.
#
# Choisissez le modèle selon la RAM de la machine :
#   "llama3.2:1b"  -> ~2 Go RAM   (machines tres modestes, le plus leger)
#   "llama3.2:3b"  -> ~5 Go RAM   (bon compromis, defaut)
#   "qwen3:1.7b"   -> ~3 Go RAM   (meilleur multilingue / francais, rapide)
#
# Prerequis (UNE fois) :  ollama pull llama3.2:3b

from llama_index.llms.ollama import Ollama

MODEL = "llama3.2:3b"   # <-- changez ce seul mot pour tester un autre modele

llm = Ollama(
    model=MODEL,
    request_timeout=300.0,   # CPU lent => on laisse du temps
    temperature=0.7,
)

# Test rapide : verifie qu'Ollama tourne et que le modele repond
print(f"🤖 Modele local : {MODEL}")
print("Reponse test :", llm.complete("Reponds juste par : OK"))

# NB pour qwen3 : il "reflechit" et peut afficher des balises <think>...</think>.
# Pour les supprimer, ajoutez " /no_think" a la fin de vos requetes.


C:\Users\mbial\AppData\Local\Temp\ipykernel_16904\1301032461.py:5: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=1, api_key=os.getenv("OPENAI_API_KEY"))


In [3]:
print("🔧 Initialisation des embeddings HuggingFace (open source, ~130 Mo)...")
# bge-small : tres leger, tourne sur CPU. Telecharge UNE fois puis mis en cache
# (~/.cache/huggingface). Pour des donnees en FRANCAIS, essayez plutot :
#   model_name="intfloat/multilingual-e5-small"
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    device="cpu",
)
print("✅ Embeddings prets")


🔧 Initializing HuggingFace embeddings...


In [4]:
Settings.llm = llm
Settings.embed_model = embed_model

# Reglages "mode leger" pour petites machines : on limite la taille du contexte,
# la longueur des reponses et la taille des fragments -> plus rapide, moins de RAM.
Settings.context_window = 2048
Settings.num_output = 256
Settings.chunk_size = 256

print("✅ LLM et embeddings configures (mode leger, 100% local)")


✅  LLM and embeddings configured!


In [5]:
# Sample data for the lab - AI/ML focused documents
SAMPLE_DOCUMENTS = [
    "Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.",
    "Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.",
    "Natural language processing enables computers to understand, interpret, and generate human language.",
    "Computer vision allows machines to interpret and understand visual information from the world.",
    "Reinforcement learning is a type of machine learning where agents learn to make decisions through rewards and penalties.",
    "Supervised learning uses labeled training data to learn a mapping from inputs to outputs.",
    "Unsupervised learning finds hidden patterns in data without labeled examples.",
    "Transfer learning leverages knowledge from pre-trained models to improve performance on new tasks.",
    "Generative AI can create new content including text, images, code, and more.",
    "Large language models are trained on vast amounts of text data to understand and generate human-like text."
]

# Consistent query examples used throughout the lab
DEMO_QUERIES = {
    "basic": "What is machine learning?",
    "technical": "neural networks deep learning", 
    "learning_types": "different types of learning",
    "advanced": "How do neural networks work in deep learning?",
    "applications": "What are the applications of AI?",
    "comprehensive": "What are the main approaches to machine learning?",
    "specific": "supervised learning techniques"
}

print(f"📄 Loaded {len(SAMPLE_DOCUMENTS)} sample documents")
print(f"🔍 Prepared {len(DEMO_QUERIES)} consistent demo queries")
for i, doc in enumerate(SAMPLE_DOCUMENTS[:3], 1):
    print(f"{i}. {doc}")
print("...")

📄 Loaded 10 sample documents
🔍 Prepared 7 consistent demo queries
1. Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
2. Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.
3. Natural language processing enables computers to understand, interpret, and generate human language.
...


In [6]:
class AdvancedRetrieversLab:
    def __init__(self):
        print("🚀 Initializing Advanced Retrievers Lab...")
        self.documents = [Document(text=text) for text in SAMPLE_DOCUMENTS]
        self.nodes = SentenceSplitter().get_nodes_from_documents(self.documents)
        
        print("📊 Creating indexes...")
        # Create various indexes
        self.vector_index = VectorStoreIndex.from_documents(self.documents)
        self.document_summary_index = DocumentSummaryIndex.from_documents(self.documents)
        self.keyword_index = KeywordTableIndex.from_documents(self.documents)
        
        print("✅ Advanced Retrievers Lab Initialized!")
        print(f"📄 Loaded {len(self.documents)} documents")
        print(f"🔢 Created {len(self.nodes)} nodes")

# Initialize the lab
lab = AdvancedRetrieversLab()

🚀 Initializing Advanced Retrievers Lab...
📊 Creating indexes...
current doc id: 1eeccfde-d128-463b-a4d2-28ad3d614416
current doc id: 20f385f8-f313-4c8f-a9e8-abe208629661
current doc id: 53bca882-7d52-43a8-b6e0-85319b3b2d0a
current doc id: 3e3eb071-6c99-47e4-b538-9e81716bd1a4
current doc id: af84f669-9baf-480f-a834-2acbc9f7648e
current doc id: aae32bdf-eac9-4312-9daa-7c871aabcaf2
current doc id: 00db9c0b-63be-4e08-a312-ecec837cec4c
current doc id: 563a9c60-5927-422a-9e43-263b0f8668ce
current doc id: 18c8c7a7-5dde-4f08-82b4-b9cd22828a42
current doc id: e3efcd02-115b-4654-a027-b9b734c064d9
✅ Advanced Retrievers Lab Initialized!
📄 Loaded 10 documents
🔢 Created 10 nodes


## 1. Vector Index Retriever - La Fondation

Le "Vector Index Retriever" utilise des "embeddings" (plongements) vectoriels pour trouver du contenu sémantiquement lié. Cela le rend idéal pour la recherche d'ordre général et très utilisé dans les pipelines de génération augmentée par récupération (RAG).



**Comment ça fonctionne :**
- Les documents sont divisés en nœuds et transformés en vecteurs (embeddings) à l'aide du modèle de configuration choisi.
- La requête est convertie en un vecteur d'embedding.
- Le système renvoie les nœuds classés par similitude cosinus (cosine similarity) par rapport au vecteur de la requête.
- Par défaut, il génère les embeddings par lots de 2048 nœuds.

**Quand l'utiliser :**
- Recherche sémantique d'ordre général (cas d'utilisation le plus courant).
- Recherche de contenu conceptuellement lié, basé sur le sens plutôt que sur des mots-clés exacts.
- Pipelines RAG où la compréhension sémantique est cruciale.
- Lorsque la correspondance exacte par mots-clés n'est pas l'exigence principale.

**Caractéristiques clés (sources faisant autorité) :**
- **Stocke les embeddings pour chaque fragment de document** (base du VectorStoreIndex).
- **Idéal pour la récupération sémantique** basée sur le sens et le contexte.
- **Couramment utilisé dans les pipelines de LLM** pour la génération augmentée par récupération.

**Points forts :**
- Excellente compréhension sémantique et sensibilité au contexte.
- Gère efficacement les synonymes et les concepts apparentés.
- Fonctionne très bien avec les requêtes en langage naturel.

**Limites :**
- Peut rater des correspondances par mots-clés exacts lorsque des termes spécifiques sont cruciaux.
- Nécessite un bon modèle d'embedding pour des performances optimales.
- Peut être gourmand en ressources de calcul pour les très grandes collections de documents.

In [7]:
print("=" * 60)
print("1. VECTOR INDEX RETRIEVER")
print("=" * 60)

# Basic vector retriever
vector_retriever = VectorIndexRetriever(
    index=lab.vector_index,
    similarity_top_k=3
)

# Alternative creation method
alt_retriever = lab.vector_index.as_retriever(similarity_top_k=3)

query = DEMO_QUERIES["basic"]  # "What is machine learning?"
nodes = vector_retriever.retrieve(query)

print(f"Query: {query}")
print(f"Retrieved {len(nodes)} nodes:")
for i, node in enumerate(nodes, 1):
    print(f"{i}. Score: {node.score:.4f}")
    print(f"   Text: {node.text[:100]}...")
    print()

1. VECTOR INDEX RETRIEVER
Query: What is machine learning?
Retrieved 3 nodes:
1. Score: 0.8700
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Score: 0.7644
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...

3. Score: 0.6979
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....



In [8]:
type(nodes)

list

## 2. BM25 Retriever - Recherche Avancée par Mots-Clés

Le BM25 est une méthode de récupération basée sur les mots-clés qui améliore le TF-IDF en corrigeant certaines de ses limites majeures. Il est largement utilisé dans les systèmes de recherche de production, notamment Elasticsearch et Apache Lucene.

### Comprendre le TF-IDF : La Fondation

Avant d'aborder le BM25, comprenons le **TF-IDF** (Term Frequency-Inverse Document Frequency), sur lequel le BM25 est bâti :

**Fréquence de Terme (TF)** : Mesure la fréquence d'apparition d'un mot dans un document.
- Exemple : Si "neural" apparaît 3 fois dans un document de 100 mots, TF = 3/100 = 0,03.

**Fréquence Inverse de Document (IDF)** : Mesure la rareté d'un mot sur l'ensemble des documents.
- Exemple : Si "neural" n'apparaît que dans 2 documents sur 1000, IDF = log(1000/2) = 6,21.
- Les mots courants comme "le" ont un IDF faible ; les termes techniques rares ont un IDF élevé.

**Score TF-IDF** : TF × IDF
- Met en avant les mots qui sont fréquents dans un document mais rares dans le reste de la collection.
- Développé par Karen Spärck Jones, pionnière du concept de spécificité des termes.

### Comment le BM25 améliore le TF-IDF



**Améliorations clés du BM25 :**

1. **Saturation de la fréquence des termes** : Le BM25 réduit l'impact des termes répétés.
   - Problème : Dans le TF-IDF, si un mot apparaît 100 fois au lieu de 10, le score augmente de façon linéaire.
   - Solution : Le BM25 utilise une fonction de saturation qui plafonne après une certaine fréquence.

2. **Normalisation de la longueur des documents** : Le BM25 ajuste le score selon la longueur du document.
   - Problème : Dans le TF-IDF, les documents longs sont injustement favorisés.
   - Solution : Le BM25 normalise les scores en fonction de la longueur du document par rapport à la moyenne.

3. **Paramètres réglables** : Permet un ajustement précis selon le type de contenu.
   - $k_1 \approx 1,2$ : Contrôle la saturation de la fréquence (vitesse à laquelle le score plafonne).
   - $b \approx 0,75$ : Contrôle la normalisation de la longueur (0 = aucune, 1 = totale).

### Quand utiliser le BM25

**Idéal pour :**
- La documentation technique où les termes exacts sont cruciaux.
- Les documents juridiques avec une terminologie spécifique.
- Les catalogues de produits avec des spécifications précises.
- Les articles académiques avec un vocabulaire spécialisé.
- Les applications nécessitant une récupération par mots-clés plutôt que par similitude sémantique.

**Avantages :**
- Excellente précision pour les correspondances exactes de termes.
- Performance de calcul rapide.
- Efficacité prouvée dans les systèmes en production.
- Aucun entraînement requis (contrairement aux approches neuronales).
- Mécanisme de scoring interprétable.

**Limites :**
- Aucune compréhension sémantique (ne gère pas les synonymes).
- Difficultés avec les fautes de frappe et les variantes.
- Compréhension limitée du contexte.
- Nécessite un réglage minutieux des paramètres pour une performance optimale.

In [8]:
print("=" * 60)
print("2. BM25 RETRIEVER")
print("=" * 60)

try:
    import Stemmer
    
    # Create BM25 retriever with default parameters
    bm25_retriever = BM25Retriever.from_defaults(
        nodes=lab.nodes,
        similarity_top_k=3,
        stemmer=Stemmer.Stemmer("english"),
        language="english"
    )
    
    query = DEMO_QUERIES["technical"]  # "neural networks deep learning"
    nodes = bm25_retriever.retrieve(query)
    
    print(f"Query: {query}")
    print("BM25 analyzes exact keyword matches with sophisticated scoring")
    print(f"Retrieved {len(nodes)} nodes:")
    
    for i, node in enumerate(nodes, 1):
        score = node.score if hasattr(node, 'score') and node.score else 0
        print(f"{i}. BM25 Score: {score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        
        # Highlight which query terms appear in the text
        text_lower = node.text.lower()
        query_terms = query.lower().split()
        found_terms = [term for term in query_terms if term in text_lower]
        if found_terms:
            print(f"   → Found terms: {found_terms}")
        print()
    
    print("BM25 vs TF-IDF Comparison:")
    print("TF-IDF Problem: Linear term frequency scaling")
    print("  Example: 10 occurrences → score of 10, 100 occurrences → score of 100")
    print("BM25 Solution: Saturation function")
    print("  Example: 10 occurrences → high score, 100 occurrences → slightly higher score")
    print()
    print("TF-IDF Problem: No document length consideration")
    print("  Example: Long documents dominate results")
    print("BM25 Solution: Length normalization (b parameter)")
    print("  Example: Scores adjusted based on document length vs. average")
    print()
    print("Key BM25 Parameters:")
    print("- k1 ≈ 1.2: Term frequency saturation (how quickly scores plateau)")
    print("- b ≈ 0.75: Document length normalization (0=none, 1=full)")
    print("- IDF weighting: Rare terms get higher scores")
        
except ImportError:
    print("⚠️ BM25Retriever requires 'pip install PyStemmer'")
    print("Demonstrating BM25 concepts with fallback vector search...")
    
    fallback_retriever = lab.vector_index.as_retriever(similarity_top_k=3)
    query = DEMO_QUERIES["technical"]
    nodes = fallback_retriever.retrieve(query)
    
    print(f"Query: {query}")
    print("(Using vector fallback to demonstrate BM25 concepts)")
    
    for i, node in enumerate(nodes, 1):
        print(f"{i}. Vector Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        
        # Demonstrate TF-IDF concept manually
        text_lower = node.text.lower()
        query_terms = query.lower().split()
        found_terms = [term for term in query_terms if term in text_lower]
        
        if found_terms:
            print(f"   → BM25 would boost this result for terms: {found_terms}")
        print()
    
    print("BM25 Concept Demonstration:")
    print("1. TF-IDF Foundation:")
    print("   - Term Frequency: How often words appear in document")
    print("   - Inverse Document Frequency: How rare words are across collection")
    print("   - TF-IDF = TF × IDF (balances frequency vs rarity)")
    print()
    print("2. BM25 Improvements:")
    print("   - Saturation: Prevents over-scoring repeated terms")
    print("   - Length normalization: Prevents long document bias")
    print("   - Tunable parameters: k1 (saturation) and b (length adjustment)")
    print()
    print("3. Real-world Usage:")
    print("   - Elasticsearch default scoring function")
    print("   - Apache Lucene/Solr standard")
    print("   - Used in 83\% of text-based recommender systems")
    print("   - Developed by Robertson & Spärck Jones at City University London")

2. BM25 RETRIEVER
Query: neural networks deep learning
BM25 analyzes exact keyword matches with sophisticated scoring
Retrieved 3 nodes:
1. BM25 Score: 2.5203
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...
   → Found terms: ['neural', 'networks', 'deep', 'learning']

2. BM25 Score: 0.3372
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...
   → Found terms: ['learning']

3. BM25 Score: 0.3024
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...
   → Found terms: ['learning']

BM25 vs TF-IDF Comparison:
TF-IDF Problem: Linear term frequency scaling
  Example: 10 occurrences → score of 10, 100 occurrences → score of 100
BM25 Solution: Saturation function
  Example: 10 occurrences → high score, 100 occurrences → slightly higher score

TF-IDF Problem: No document length consideration
  Example: Long docume

## 3. Document Summary Index Retrievers (Récupérateurs par Index de Résumés)

Les récupérateurs par index de résumés de documents utilisent les résumés plutôt que les documents réels pour trouver le contenu pertinent, ce qui les rend efficaces pour les grandes collections. **Ils renvoient les documents originaux, et non leurs résumés.**



**Comment ça fonctionne (selon les sources faisant autorité) :**
- **Génère et stocke des résumés de documents** au moment de l'indexation.
- **Utilise les résumés pour filtrer les documents** avant de récupérer le contenu complet.
- **Processus en deux étapes** : utilise d'abord les résumés pour filtrer les documents, puis renvoie le contenu complet des documents sélectionnés.
- **Particulièrement utile pour les corpus volumineux et diversifiés** qui ne peuvent pas tenir dans la fenêtre de contexte d'un LLM.

**Deux options de récupération :**
1. **DocumentSummaryIndexLLMRetriever :**
   - Utilise un grand modèle de langage (LLM) pour analyser la requête par rapport aux résumés de documents.
   - Offre une sélection intelligente des documents, mais peut être plus long et plus coûteux.
   - Idéal pour les requêtes complexes nécessitant une compréhension nuancée.

2. **DocumentSummaryIndexEmbeddingRetriever :**
   - Utilise la similitude sémantique entre les "embeddings" de la requête et ceux des résumés.
   - Plus rapide et plus rentable que l'approche basée sur le LLM.
   - Adapté pour une correspondance de similitude directe.

**Quand l'utiliser (selon les recommandations officielles) :**
- Grandes collections de documents où les documents traitent de sujets différents.
- Lorsque vous avez besoin d'un filtrage efficace au niveau du document avant une récupération détaillée.
- Assurance qualité (QA) multi-documents où les documents ont des thématiques distinctes.
- Ensembles de documents vastes et hétérogènes dépassant la fenêtre de contexte d'un LLM.

**Paramètres de configuration :**
- `choice_top_k` (récupérateur LLM) : Nombre de documents à sélectionner.
- `similarity_top_k` (récupérateur par embedding) : Nombre de documents à sélectionner.
- La valeur par défaut est 1 ; augmentez-la pour une récupération multi-documents.

**Point clé :** **Renvoie les documents originaux, pas leurs résumés.** Les résumés ne servent qu'au filtrage.

**Points forts :**
- Sélection de documents efficace et réduction de l'espace de recherche.
- Adapté aux collections hétérogènes traitant de sujets variés.
- Renvoie les documents originaux avec tout leur contexte intact.

**Limites :**
- Nécessite un LLM pour la génération de résumés lors de l'indexation.
- Peut perdre certains détails présents dans les documents originaux lors de la création des résumés.
- La version basée sur le LLM peut être plus lente et plus coûteuse que les autres options.

In [9]:
print("=" * 60)
print("3. DOCUMENT SUMMARY INDEX RETRIEVERS")
print("=" * 60)

# LLM-based document summary retriever
doc_summary_retriever_llm = DocumentSummaryIndexLLMRetriever(
    lab.document_summary_index,
    choice_top_k=3  # Number of documents to select
)

# Embedding-based document summary retriever  
doc_summary_retriever_embedding = DocumentSummaryIndexEmbeddingRetriever(
    lab.document_summary_index,
    similarity_top_k=3  # Number of documents to select
)

query = DEMO_QUERIES["learning_types"]  # "different types of learning"

print(f"Query: {query}")

print("\nA) LLM-based Document Summary Retriever:")
print("Uses LLM to select relevant documents based on summaries")
try:
    nodes_llm = doc_summary_retriever_llm.retrieve(query)
    print(f"Retrieved {len(nodes_llm)} nodes")
    for i, node in enumerate(nodes_llm[:2], 1):
        print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Document summary)")
        print(f"   Text: {node.text[:80]}...")
        print()
except Exception as e:
    print(f"LLM-based retrieval demo: {str(e)[:100]}...")

print("B) Embedding-based Document Summary Retriever:")
print("Uses vector similarity between query and document summaries")
try:
    nodes_emb = doc_summary_retriever_embedding.retrieve(query)
    print(f"Retrieved {len(nodes_emb)} nodes")
    for i, node in enumerate(nodes_emb[:2], 1):
        print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Document summary)")
        print(f"   Text: {node.text[:80]}...")
        print()
except Exception as e:
    print(f"Embedding-based retrieval demo: {str(e)[:100]}...")

print("Document Summary Index workflow:")
print("1. Generates summaries for each document using LLM")
print("2. Uses summaries to select relevant documents")
print("3. Returns full content from selected documents")

3. DOCUMENT SUMMARY INDEX RETRIEVERS
Query: different types of learning

A) LLM-based Document Summary Retriever:
Uses LLM to select relevant documents based on summaries
Retrieved 3 nodes
1. Score: 9.0000
   Text: Reinforcement learning is a type of machine learning where agents learn to make ...

2. Score: 7.0000
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to...

B) Embedding-based Document Summary Retriever:
Uses vector similarity between query and document summaries
Retrieved 3 nodes
1. (Document summary)
   Text: Unsupervised learning finds hidden patterns in data without labeled examples....

2. (Document summary)
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to...

Document Summary Index workflow:
1. Generates summaries for each document using LLM
2. Uses summaries to select relevant documents
3. Returns full content from selected documents


## 4. Auto Merging Retriever - Préservation du Contexte Hiérarchique

L'Auto Merging Retriever est conçu pour préserver le contexte dans les documents longs en utilisant une structure hiérarchique. **Il utilise un découpage (chunking) hiérarchique pour diviser les documents en nœuds parents et enfants ; si suffisamment de nœuds enfants issus d'un même parent sont récupérés, le système renvoie le nœud parent à la place.**



**Comment ça fonctionne (selon les sources faisant autorité) :**
- **Utilise le découpage hiérarchique** pour segmenter les documents en nœuds parents et enfants.
- **Récupère le parent si suffisamment d'enfants correspondent** : une logique de fusion intelligente.
- **Préserve le contexte dans les documents longs** en consolidant le contenu connexe.
- **Stockage double** : Les petits fragments enfants sont indexés dans le "vector store" pour une correspondance précise, tandis que les fragments parents plus larges sont stockés dans le "docstore".

**Schéma de comportement clé :**
- Les fragments enfants permettent une correspondance précise pour des requêtes spécifiques.
- Lorsque plusieurs fragments enfants d'un même parent sont récupérés, le système renvoie le fragment parent.
- Cela **aide à consolider le contenu lié et à préserver un contexte plus large**.

**Quand l'utiliser (selon les recommandations officielles) :**
- Documents longs où les petits fragments perdent le contexte environnant important.
- Documents juridiques, articles de recherche ou spécifications techniques nécessitant la préservation du contexte.
- Lorsque vous avez besoin à la fois d'une correspondance précise et d'un contexte complet.
- Documents ayant une structure hiérarchique naturelle (sections, sous-sections).

**Configuration :**
- `chunk_sizes` : Liste des tailles de fragments, du plus grand au plus petit (ex: [512, 256, 128]).
- `chunk_overlap` : Chevauchement entre les fragments pour maintenir la continuité.
- Le contexte de stockage gère à la fois le "vector store" (nœuds enfants) et le "docstore" (nœuds parents).

**Points forts :**
- Préserve automatiquement le contexte sans intervention manuelle.
- Réduit la fragmentation de l'information dans les documents longs.
- Fusion intelligente basée sur les schémas de récupération.
- Maintient une capacité de recherche granulaire tout en fournissant un contexte étendu.

**Limites :**
- Configuration plus complexe par rapport aux récupérateurs de base.
- Nécessite une structure de document hiérarchique pour être efficace.
- Surcharge de stockage plus élevée due aux multiples niveaux de fragments.
- Peut ne pas convenir aux documents très courts.

*Source : https://docs.llamaindex.ai/en/stable/examples/retrievers/auto_merging_retriever/*

In [10]:
print("=" * 60)
print("4. AUTO MERGING RETRIEVER")
print("=" * 60)

# Create hierarchical nodes
node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[512, 256, 128]
)

hier_nodes = node_parser.get_nodes_from_documents(lab.documents)

# Create storage context with all nodes
from llama_index.core import StorageContext
from llama_index.core.storage.docstore import SimpleDocumentStore

docstore = SimpleDocumentStore()
docstore.add_documents(hier_nodes)

storage_context = StorageContext.from_defaults(docstore=docstore)

# Create base index
base_index = VectorStoreIndex(hier_nodes, storage_context=storage_context)
base_retriever = base_index.as_retriever(similarity_top_k=6)

# Create auto-merging retriever
auto_merging_retriever = AutoMergingRetriever(
    base_retriever, 
    storage_context,
    verbose=True
)

query = DEMO_QUERIES["advanced"]  # "How do neural networks work in deep learning?"
nodes = auto_merging_retriever.retrieve(query)

print(f"Query: {query}")
print(f"Auto-merged to {len(nodes)} nodes")
for i, node in enumerate(nodes[:3], 1):
    print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Auto-merged)")
    print(f"   Text: {node.text[:120]}...")
    print()

4. AUTO MERGING RETRIEVER
Query: How do neural networks work in deep learning?
Auto-merged to 2 nodes
1. Score: 0.8570
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in data....

2. Score: 0.6956
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....



## 5. Recursive Retriever - Suivi de Références Multi-Niveaux

Le "Recursive Retriever" est **conçu pour suivre les relations entre les nœuds en utilisant des références**. **Il peut suivre des références d'un nœud à un autre, comme des citations dans des articles académiques ou d'autres liens de métadonnées**, ce qui lui permet de **récupérer du contenu lié à travers différents documents ou couches d'abstraction**.



**Comment ça fonctionne (selon les sources faisant autorité) :**
- **Suit les références de nœuds** : parcourt les relations pour trouver le contenu référencé.
- **Prend en charge les liens entre fragments et métadonnées** : gère différents types de références.
- **Navigation multi-niveaux** : peut exécuter des sous-requêtes sur des récupérateurs ou des moteurs de requête (query engines) référencés.
- **Construction de réseau** : crée un réseau de récupérateurs interconnectés qui peuvent se référencer mutuellement.

**Types de références pris en charge :**
1. **Références de fragments (Chunks)** : de petits fragments enfants renvoient à des fragments parents plus larges pour un contexte supplémentaire.
2. **Références de métadonnées** : des résumés ou des questions générées renvoient à des fragments de contenu plus vastes, comme des citations dans des articles académiques.

**Quand l'utiliser (selon les recommandations officielles) :**
- **Articles académiques avec citations** et références étendues.
- **Documents de recherche** où vous devez récupérer du contenu pertinent à partir d'articles cités.
- Documentation contenant des références croisées et du contenu lié.
- Bases de connaissances avec des informations interconnectées.
- Lorsque les nœuds font référence à des données structurées (tableaux, bases de données, autres documents).

**Configuration :**
- `retriever_dict` : associe les identifiants de nœuds ou les clés à des récupérateurs spécifiques.
- `query_engine_dict` : associe des clés à des moteurs de requête pour les sous-requêtes.
- Les métadonnées des nœuds peuvent contenir des références vers d'autres nœuds ou structures de données.

**Capacité clé :** **Récupère du contenu lié à travers les documents** en suivant des chaînes de références.

**Points forts :**
- Suit des relations complexes et permet un raisonnement en plusieurs étapes.
- Offre une couverture complète sur l'ensemble des documents liés.
- Excellent pour la gestion de systèmes d'information interconnectés.
- Peut parcourir automatiquement plusieurs niveaux de références.

**Limites :**
- Nécessite une configuration minutieuse des relations entre les nœuds.
- Peut être coûteux en ressources de calcul pour les chaînes de références profondes.
- Débogage complexe lorsque les chaînes de références sont étendues.
- Risque de récupérer trop de contenu lié s'il n'est pas correctement configuré.

*Source : https://docs.llamaindex.ai/en/stable/examples/retrievers/recurisve_retriever_nodes_braintrust/*

In [11]:
print("=" * 60)
print("5. RECURSIVE RETRIEVER")
print("=" * 60)

# Create documents with references
docs_with_refs = []
for i, doc in enumerate(lab.documents):
    # Add reference metadata
    ref_doc = Document(
        text=doc.text,
        metadata={
            "doc_id": f"doc_{i}",
            "references": [f"doc_{j}" for j in range(len(lab.documents)) if j != i][:2]
        }
    )
    docs_with_refs.append(ref_doc)

# Create index with referenced documents
ref_index = VectorStoreIndex.from_documents(docs_with_refs)

# Create retriever mapping
retriever_dict = {
    f"doc_{i}": ref_index.as_retriever(similarity_top_k=1)
    for i in range(len(docs_with_refs))
}

# Base retriever
base_retriever = ref_index.as_retriever(similarity_top_k=2)

# Add the root retriever to the dictionary
retriever_dict["vector"] = base_retriever

# Recursive retriever
recursive_retriever = RecursiveRetriever(
    "vector",
    retriever_dict=retriever_dict,
    query_engine_dict={},
    verbose=True
)

query = DEMO_QUERIES["applications"]  # "What are the applications of AI?"
try:
    nodes = recursive_retriever.retrieve(query)
    print(f"Query: {query}")
    print(f"Recursively retrieved {len(nodes)} nodes")
    for i, node in enumerate(nodes[:3], 1):
        print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Recursive)")
        print(f"   Text: {node.text[:100]}...")
        print()
except Exception as e:
    print(f"Query: {query}")
    print(f"Recursive retriever demo: {str(e)}")
    print("Note: Recursive retriever requires specific node reference setup")
    
    # Fallback to basic retrieval for demonstration
    print("\nFalling back to basic retrieval demonstration...")
    base_nodes = base_retriever.retrieve(query)
    for i, node in enumerate(base_nodes[:2], 1):
        print(f"{i}. Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()

5. RECURSIVE RETRIEVER
Retrieving with query id None: What are the applications of AI?
Retrieving text node: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
Retrieving text node: Natural language processing enables computers to understand, interpret, and generate human language.
Query: What are the applications of AI?
Recursively retrieved 2 nodes
1. Score: 0.6907
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Score: 0.6500
   Text: Natural language processing enables computers to understand, interpret, and generate human language....



## 6. Query Fusion Retriever - Amélioration Multi-Requêtes avec Fusion Avancée

Le "Query Fusion Retriever" **combine les résultats de différents récupérateurs** (tels que les méthodes vectorielles et par mots-clés) et **génère optionnellement plusieurs variantes d'une requête à l'aide d'un LLM pour améliorer la couverture**. **Les résultats sont ensuite fusionnés à l'aide de stratégies de fusion** pour augmenter le rappel (recall).



**Comment ça fonctionne (selon les sources faisant autorité) :**
- **Combine les résultats de plusieurs récupérateurs** : par exemple, des méthodes basées sur les vecteurs et sur les mots-clés.
- **Prend en charge plusieurs variantes de requête** : génère différentes formulations d'une même question.
- **Utilise des stratégies de fusion pour améliorer le rappel** : techniques de regroupement sophistiquées.
- **Couverture améliorée** : réduit l'impact de la formulation de la requête sur les résultats finaux.

**Capacités centrales :**
1. **Support multi-récupérateurs** : combine les résultats provenant de différentes sources.
2. **Génération de variantes de requête** : génère optionnellement plusieurs reformulations via un LLM.
3. **Stratégies de fusion** : fusionne les résultats à l'aide de techniques avancées.

**Stratégies de fusion prises en charge (selon les sources faisant autorité) :**
1. **Reciprocal Rank Fusion (RRF)** : **Combine les classements à travers les requêtes** ; robuste car elle ne dépend pas de l'ampleur des scores.
2. **Relative Score Fusion** : **Normalise les scores au sein de chaque ensemble de résultats** ; préserve la confiance relative de chaque récupérateur.
3. **Distribution-Based Fusion** : **Utilise la normalisation statistique** ; idéal pour gérer la variabilité des scores.

**Quand l'utiliser (selon les recommandations officielles) :**
- Questions-réponses générales où vous voulez combiner pertinence sémantique et correspondance par mots-clés.
- Requêtes complexes ou ambiguës qui peuvent bénéficier de plusieurs formulations.
- Lorsque la formulation de la requête impacte considérablement les résultats.
- Scénarios de recherche documentaire et exploratoire.
- Lorsque les utilisateurs fournissent des requêtes imprécises ou peu claires.

**Configuration :**
- `num_queries` : Nombre de variantes de requêtes à générer (par défaut : 4).
- `mode` : Stratégie de fusion ("reciprocal_rerank", "relative_score", "dist_based_score").
- `similarity_top_k` : Nombre de résultats à récupérer par requête.
- `use_async` : Active le traitement asynchrone pour de meilleures performances.

**Avantage clé :** **Utilise des stratégies de fusion telles que la "reciprocal rank fusion" ou la "relative score fusion"** pour combiner intelligemment les résultats.

**Points forts :**
- Rappel amélioré grâce aux multiples formulations de requêtes.
- Gère efficacement les variations de langage.
- Réduit la sensibilité à la formulation initiale.
- Combine les forces de différentes méthodes de récupération.

**Limites :**
- Coût de calcul plus élevé dû à la multiplicité des récupérateurs/requêtes.
- Nécessite un LLM pour la génération de requêtes (coût supplémentaire).
- Peut introduire du "bruit" si les stratégies de fusion ne sont pas bien réglées.
- Configuration et installation plus complexes.

In [14]:
print("=" * 60)
print("6. QUERY FUSION RETRIEVER - OVERVIEW")
print("=" * 60)

# Create base retriever
base_retriever = lab.vector_index.as_retriever(similarity_top_k=3)

query = DEMO_QUERIES["comprehensive"]  # "What are the main approaches to machine learning?"
print(f"Query: {query}")
print("QueryFusionRetriever generates multiple query variations and fuses results")
print("using one of three sophisticated fusion modes.")

print("\nOverview of Fusion Modes:")
print("1. RECIPROCAL_RERANK: Uses reciprocal rank fusion (most robust)")
print("2. RELATIVE_SCORE: Preserves score magnitudes (most interpretable)")  
print("3. DIST_BASED_SCORE: Statistical normalization (most sophisticated)")

print("\nDemonstration workflow:")
print("Each subsection below explores one fusion mode in detail with:")
print("- Theoretical explanation of the fusion method")
print("- Live demonstration using QueryFusionRetriever")
print("- Manual implementation showing the underlying mathematics")
print("- Use case recommendations and trade-offs")

print(f"\nUsing consistent test query throughout: '{query}'")
print("This allows direct comparison of how each fusion mode handles the same input.")

print("\nProceed to subsections 6.1, 6.2, and 6.3 for detailed demonstrations...")

6. QUERY FUSION RETRIEVER - OVERVIEW
Query: What are the main approaches to machine learning?
QueryFusionRetriever generates multiple query variations and fuses results
using one of three sophisticated fusion modes.

Overview of Fusion Modes:
1. RECIPROCAL_RERANK: Uses reciprocal rank fusion (most robust)
2. RELATIVE_SCORE: Preserves score magnitudes (most interpretable)
3. DIST_BASED_SCORE: Statistical normalization (most sophisticated)

Demonstration workflow:
Each subsection below explores one fusion mode in detail with:
- Theoretical explanation of the fusion method
- Live demonstration using QueryFusionRetriever
- Manual implementation showing the underlying mathematics
- Use case recommendations and trade-offs

Using consistent test query throughout: 'What are the main approaches to machine learning?'
This allows direct comparison of how each fusion mode handles the same input.

Proceed to subsections 6.1, 6.2, and 6.3 for detailed demonstrations...


### 6.1 Mode Reciprocal Rank Fusion (RRF)

La "Reciprocal Rank Fusion" (RRF) est la méthode de fusion la plus robuste du `QueryFusionRetriever`. Elle est conçue pour combiner les listes classées de plusieurs variantes de requêtes en utilisant l'inverse des rangs, ce qui réduit l'impact des valeurs aberrantes et fournit des résultats de fusion stables.



**Comment ça fonctionne au sein du QueryFusionRetriever :**
- Génère plusieurs variantes de requête (ex: "approches d'apprentissage automatique", "techniques de ML", "algorithmes d'apprentissage").
- Récupère les résultats pour chaque variante de requête.
- Calcule le score de rang réciproque : $1 / (rang + k)$, où $k$ est généralement égal à 60.
- Additionne les scores de rang réciproque de toutes les variantes de requête pour chaque document.
- Re-classe les documents selon les scores RRF combinés.

**Formule mathématique :**
$$RRF\_score(d) = \sum_{i} \frac{1}{rank_i(d) + k}$$
Où :
- $d$ est un document.
- $rank_i(d)$ est le rang du document $d$ dans les résultats de la variante de requête $i$.
- $k$ est une constante (généralement 60) qui contrôle le comportement de la fusion.

**Pourquoi la RRF fonctionne-t-elle bien pour la fusion de requêtes ?**
- **Invariance d'échelle** : Fonctionne indépendamment des plages de scores des résultats de chaque requête individuelle.
- **Robustesse aux valeurs aberrantes** : La fonction réciproque réduit l'impact des classements extrêmes.
- **Agnostique à la requête** : Ne dépend pas des formulations spécifiques des requêtes.
- **Efficacité prouvée** : Concept bien établi dans la recherche en recherche d'information (IR).

**Quand utiliser le mode RRF :**
- Choix par défaut pour la plupart des scénarios de fusion de requêtes.
- Lorsque les variantes de requêtes peuvent avoir des qualités de résultats très différentes.
- Lorsque vous souhaitez un comportement de fusion stable et prévisible.
- Pour les systèmes de production nécessitant des performances constantes.

**Avantages :**
- Méthode de fusion la plus stable pour différents types de requêtes.
- Aucun réglage de paramètres requis au-delà du standard $k=60$.
- Gère gracieusement les nombres variables de résultats par variante de requête.
- Efficace sur le plan informatique.

**Limites :**
- Perd l'information sur le score absolu des requêtes individuelles.
- Traite toutes les variantes de requêtes de manière égale (pas de pondération).
- Peut ne pas exploiter efficacement les différences d'amplitude de score.

*Source : https://docs.llamaindex.ai/en/stable/examples/retrievers/reciprocal_rerank_fusion/*

In [15]:
print("=" * 60)
print("6.1 RECIPROCAL RANK FUSION MODE DEMONSTRATION")
print("=" * 60)

# Create QueryFusionRetriever with RRF mode
base_retriever = lab.vector_index.as_retriever(similarity_top_k=5)

print("Testing QueryFusionRetriever with reciprocal_rerank mode:")
print("This demonstrates how RRF works within the query fusion framework")

# Use the same query for consistency across all fusion modes
query = DEMO_QUERIES["comprehensive"]  # "What are the main approaches to machine learning?"

try:
    # Create query fusion retriever with RRF mode
    rrf_query_fusion = QueryFusionRetriever(
        [base_retriever],
        similarity_top_k=3,
        num_queries=3,
        mode="reciprocal_rerank",
        use_async=False,
        verbose=True
    )
    
    print(f"\nQuery: {query}")
    print("QueryFusionRetriever will:")
    print("1. Generate query variations using LLM")
    print("2. Retrieve results for each variation")
    print("3. Apply Reciprocal Rank Fusion")
    
    nodes = rrf_query_fusion.retrieve(query)
    
    print(f"\nRRF Query Fusion Results:")
    for i, node in enumerate(nodes, 1):
        print(f"{i}. Final RRF Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()
    
    print("RRF Benefits in Query Fusion Context:")
    print("- Automatically handles query variations of different quality")
    print("- No bias toward queries that return higher raw scores")
    print("- Stable performance across diverse query formulations")
    
except Exception as e:
    print(f"QueryFusionRetriever error: {e}")
    print("Demonstrating RRF concept manually with query variations...")
    
    # Manual demonstration with query variations derived from the main query
    query_variations = [
        DEMO_QUERIES["comprehensive"],  # Original query
        "machine learning approaches and methods",
        "different ML techniques and algorithms"
    ]
    
    print("Manual RRF with Query Variations:")
    all_results = {}
    
    for i, query_var in enumerate(query_variations):
        print(f"\nQuery variation {i+1}: {query_var}")
        nodes = base_retriever.retrieve(query_var)
        
        # Apply RRF scoring
        for rank, node in enumerate(nodes):
            node_id = node.node.node_id
            if node_id not in all_results:
                all_results[node_id] = {
                    'node': node,
                    'rrf_score': 0,
                    'query_ranks': []
                }
            
            # Calculate RRF contribution: 1 / (rank + k)
            k = 60  # Standard RRF parameter
            rrf_contribution = 1.0 / (rank + 1 + k)
            all_results[node_id]['rrf_score'] += rrf_contribution
            all_results[node_id]['query_ranks'].append((i, rank + 1))
    
    # Sort by final RRF score
    sorted_results = sorted(
        all_results.values(), 
        key=lambda x: x['rrf_score'], 
        reverse=True
    )
    
    print(f"\nCombined RRF Results (top 3):")
    for i, result in enumerate(sorted_results[:3], 1):
        print(f"{i}. Final RRF Score: {result['rrf_score']:.4f}")
        print(f"   Query ranks: {result['query_ranks']}")
        print(f"   Text: {result['node'].text[:100]}...")
        print()
    
    print("RRF Formula Demonstration:")
    print("For each document: RRF_score = Σ(1 / (rank + 60))")
    print("- Rank 1 in query: 1/(1+60) = 0.0164")
    print("- Rank 2 in query: 1/(2+60) = 0.0161")
    print("- Rank 3 in query: 1/(3+60) = 0.0159")
    print("Documents appearing in multiple queries get higher combined scores")

6.1 RECIPROCAL RANK FUSION MODE DEMONSTRATION
Testing QueryFusionRetriever with reciprocal_rerank mode:
This demonstrates how RRF works within the query fusion framework

Query: What are the main approaches to machine learning?
QueryFusionRetriever will:
1. Generate query variations using LLM
2. Retrieve results for each variation
3. Apply Reciprocal Rank Fusion
Generated queries:
1. Different types of machine learning algorithms
2. Comparison between supervised and unsupervised machine learning techniques

RRF Query Fusion Results:
1. Final RRF Score: 0.0492
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Final RRF Score: 0.0489
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....

3. Final RRF Score: 0.0487
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...

RRF Benefits in Query Fusion Context:
- Automatic

### 6.2 Mode Relative Score Fusion

La "Relative Score Fusion" (Fusion par Score Relatif) normalise les scores de récupération par rapport au score maximum obtenu au sein des résultats de chaque variante de requête. Cela permet une combinaison efficace lorsque l'on souhaite préserver l'information sur l'amplitude du score à travers différentes formulations de requêtes.



**Comment ça fonctionne au sein du QueryFusionRetriever :**
- Génère plusieurs variantes de requêtes à l'aide d'un LLM.
- Récupère les résultats pour chaque variante de requête.
- Normalise les scores de chaque requête en les divisant par le score maximum obtenu dans les résultats de cette même requête.
- Crée des scores compris dans l'intervalle [0, 1], où 1 représente le meilleur résultat de chaque variante de requête.
- Combine les scores normalisés en utilisant une moyenne pondérée ou une somme.

**Approche mathématique :**
$$normalized\_score_i(d) = \frac{score_i(d)}{max\_score_i}$$
$$combined\_score(d) = \sum (weight_i \times normalized\_score_i(d))$$

**Pourquoi la Relative Score Fusion est précieuse pour les variantes de requêtes :**
- **Préserve l'amplitude des scores** : Contrairement à la RRF, elle conserve l'information sur le niveau de confiance de chaque requête envers ses résultats.
- **Combinaison équitable** : Garantit qu'aucune variante de requête ne domine les autres simplement à cause d'échelles de notation différentes.
- **Résultats interprétables** : Les scores finaux reflètent la force relative à travers les différentes variantes de requêtes.
- **Pondération flexible** : Permet de donner plus de poids à certaines formulations de requêtes si nécessaire.

**Quand utiliser le mode Relative Score :**
- Lorsque vous avez confiance dans les scores de confiance du modèle d'embedding.
- Pour les requêtes où l'amplitude des scores est significative.
- Lorsque différentes variantes de requêtes doivent contribuer proportionnellement à leur niveau de confiance.
- Dans les scénarios où vous voulez comprendre pourquoi certains résultats sont classés en tête.

**Configuration au sein du QueryFusionRetriever :**
- Gère automatiquement la normalisation des scores entre les variantes de requêtes.
- Pondération égale de toutes les variantes de requêtes par défaut.
- Préserve les différences relatives de confiance du récupérateur.

**Avantages :**
- Préserve les informations précieuses sur l'amplitude des scores.
- Approche de normalisation intuitive.
- Fonctionne bien lorsque les scores du récupérateur sont fiables.
- Plus interprétable que les méthodes basées uniquement sur le rang.

**Limites :**
- Sensible aux scores aberrants (outliers) au sein des résultats d'une requête individuelle.
- Suppose que les scores du récupérateur sont significatifs et comparables.
- Peut ne pas bien gérer les mécanismes de scoring peu fiables.

*Source : https://docs.llamaindex.ai/en/stable/examples/retrievers/relative_score_dist_fusion/*

In [16]:
print("=" * 60)
print("6.2 RELATIVE SCORE FUSION MODE DEMONSTRATION")
print("=" * 60)

base_retriever = lab.vector_index.as_retriever(similarity_top_k=5)

print("Testing QueryFusionRetriever with relative_score mode:")
print("This mode preserves score magnitudes while normalizing across query variations")

# Use the same query for consistency across all fusion modes
query = DEMO_QUERIES["comprehensive"]  # "What are the main approaches to machine learning?"

try:
    # Create query fusion retriever with relative score mode
    rel_score_fusion = QueryFusionRetriever(
        [base_retriever],
        similarity_top_k=3,
        num_queries=3,
        mode="relative_score",
        use_async=False,
        verbose=False
    )
    
    print(f"\nQuery: {query}")
    print("QueryFusionRetriever with relative_score will:")
    print("1. Generate query variations")
    print("2. Normalize scores within each variation (score/max_score)")
    print("3. Combine normalized scores")
    
    nodes = rel_score_fusion.retrieve(query)
    
    print(f"\nRelative Score Fusion Results:")
    for i, node in enumerate(nodes, 1):
        print(f"{i}. Combined Relative Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()
    
    print("Relative Score Benefits in Query Fusion:")
    print("- Preserves confidence information from embedding model")
    print("- Ensures fair contribution from each query variation")
    print("- More interpretable than rank-only methods")
    
except Exception as e:
    print(f"QueryFusionRetriever error: {e}")
    print("Demonstrating Relative Score concept manually...")
    
    # Manual demonstration with query variations derived from the main query
    query_variations = [
        DEMO_QUERIES["comprehensive"],  # Original query
        "machine learning approaches and methods",
        "different ML techniques and algorithms"
    ]
    
    print("Manual Relative Score Fusion with Query Variations:")
    all_results = {}
    query_max_scores = []
    
    # Step 1: Get results and find max scores for each query
    for i, query_var in enumerate(query_variations):
        print(f"\nQuery variation {i+1}: {query_var}")
        nodes = base_retriever.retrieve(query_var)
        scores = [node.score or 0 for node in nodes]
        max_score = max(scores) if scores else 1.0
        query_max_scores.append(max_score)
        
        print(f"Max score for this query: {max_score:.4f}")
        
        # Store results with normalization info
        for node in nodes:
            node_id = node.node.node_id
            original_score = node.score or 0
            normalized_score = original_score / max_score if max_score > 0 else 0
            
            if node_id not in all_results:
                all_results[node_id] = {
                    'node': node,
                    'combined_score': 0,
                    'contributions': []
                }
            
            all_results[node_id]['combined_score'] += normalized_score
            all_results[node_id]['contributions'].append({
                'query': i,
                'original': original_score,
                'normalized': normalized_score
            })
    
    # Step 2: Sort by combined relative score
    sorted_results = sorted(
        all_results.values(),
        key=lambda x: x['combined_score'],
        reverse=True
    )
    
    print(f"\nCombined Relative Score Results (top 3):")
    for i, result in enumerate(sorted_results[:3], 1):
        print(f"{i}. Combined Score: {result['combined_score']:.4f}")
        print(f"   Score breakdown:")
        for contrib in result['contributions']:
            print(f"     Query {contrib['query']}: {contrib['original']:.3f} → {contrib['normalized']:.3f}")
        print(f"   Text: {result['node'].text[:100]}...")
        print()
    
    print("Relative Score Normalization Process:")
    print("1. For each query variation, find max_score")
    print("2. Normalize: normalized_score = original_score / max_score")
    print("3. Sum normalized scores across all query variations")
    print("4. Documents with consistently high scores across queries win")

6.2 RELATIVE SCORE FUSION MODE DEMONSTRATION
Testing QueryFusionRetriever with relative_score mode:
This mode preserves score magnitudes while normalizing across query variations

Query: What are the main approaches to machine learning?
QueryFusionRetriever with relative_score will:
1. Generate query variations
2. Normalize scores within each variation (score/max_score)
3. Combine normalized scores

Relative Score Fusion Results:
1. Combined Relative Score: 0.7012
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Combined Relative Score: 0.4743
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....

3. Combined Relative Score: 0.3650
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...

Relative Score Benefits in Query Fusion:
- Preserves confidence information from embedding model
- Ensures fair contribution from 

### 6.3 Mode Distribution-Based Score Fusion

La "Distribution-Based Score Fusion" (Fusion de scores basée sur la distribution) utilise les propriétés statistiques des distributions de scores de chaque variante de requête pour normaliser et combiner les résultats. C'est la méthode la plus sophistiquée pour gérer la variabilité et la fiabilité des scores entre différentes formulations de requêtes.



**Comment ça fonctionne au sein du QueryFusionRetriever :**
- Génère plusieurs variantes de requêtes à l'aide d'un LLM.
- Analyse la distribution statistique des scores pour chaque variante.
- Normalise les scores en utilisant des paramètres de distribution (moyenne, écart-type, centiles).
- Applique des transformations statistiques comme la normalisation par score Z (z-score) ou le classement par centiles.
- Combine les scores normalisés avec une pondération de confiance basée sur les caractéristiques de la distribution.

**Approches statistiques utilisées :**
1. **Normalisation par score Z (Z-score)** : Centre les scores autour de la moyenne avec une variance unitaire.
   - Formule : $z = (score - moyenne) / écart\_type$
   - Conversion vers un intervalle [0,1] via une fonction sigmoïde : $1 / (1 + exp(-z))$
2. **Classement par centiles** : Convertit les scores en positions centiles.
   - Formule : $centile = rang(score) / total\_des\_résultats$
3. **Normalisation sensible à la distribution** : Prend en compte la forme de la distribution des scores.
   - Utilise l'Ecart Interquartile (IQR) pour ajuster l'étalement de la distribution.
   - Gère les distributions multimodales issues des différentes variantes de requêtes.

**Pourquoi cette méthode excelle pour les variantes de requêtes :**
- **Robustesse statistique** : Tient compte de la manière dont les scores sont répartis au sein de chaque variante.
- **Pondération adaptative** : Peut pondérer les variantes de requêtes selon la confiance de leur distribution de scores.
- **Gestion des valeurs aberrantes** : Les méthodes statistiques traitent naturellement les scores extrêmes.
- **Support multimodal** : Chaque variante peut avoir des caractéristiques de distribution de scores très différentes.

**Quand utiliser le mode Distribution-Based :**
- Lorsque les variantes de requêtes produisent des distributions de scores très disparates.
- Pour les requêtes complexes où certaines variantes sont beaucoup plus fiables que d'autres.
- Lorsque vous avez besoin d'une combinaison de scores fondée sur des principes statistiques rigoureux.
- Dans des scénarios où le scoring de récupération est bruité ou peu fiable.

**Avantages :**
- Approche la plus rigoureuse statistiquement pour la fusion de requêtes.
- Gère efficacement les distributions de scores complexes.
- S'adapte aux caractéristiques propres de chaque variante de requête.
- Robuste face à divers types de variabilité et de bruit dans les scores.

**Limites :**
- Méthode de fusion la plus gourmande en ressources de calcul.
- Nécessite un nombre suffisant de résultats pour une estimation fiable de la distribution.
- Peut entraîner une sur-normalisation dans certains scénarios simples.
- Plus complexe à interpréter que les méthodes de fusion plus basiques.

*Source : https://docs.llamaindex.ai/en/stable/examples/retrievers/relative_score_dist_fusion/*

In [17]:
print("=" * 60)
print("6.3 DISTRIBUTION-BASED SCORE FUSION MODE DEMONSTRATION")
print("=" * 60)

base_retriever = lab.vector_index.as_retriever(similarity_top_k=8)

print("Testing QueryFusionRetriever with dist_based_score mode:")
print("This mode uses statistical analysis for the most sophisticated score fusion")

# Use the same query for consistency across all fusion modes
query = DEMO_QUERIES["comprehensive"]  # "What are the main approaches to machine learning?"

try:
    # Create query fusion retriever with distribution-based mode
    dist_fusion = QueryFusionRetriever(
        [base_retriever],
        similarity_top_k=3,
        num_queries=3,
        mode="dist_based_score",
        use_async=False,
        verbose=False
    )
    
    print(f"\nQuery: {query}")
    print("QueryFusionRetriever with dist_based_score will:")
    print("1. Generate query variations")
    print("2. Analyze score distributions for each variation")
    print("3. Apply statistical normalization (z-score, percentiles)")
    print("4. Combine with distribution-aware weighting")
    
    nodes = dist_fusion.retrieve(query)
    
    print(f"\nDistribution-Based Fusion Results:")
    for i, node in enumerate(nodes, 1):
        print(f"{i}. Statistically Normalized Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()
    
    print("Distribution-Based Benefits in Query Fusion:")
    print("- Accounts for score distribution differences between query variations")
    print("- Statistically robust against outliers and noise")
    print("- Adapts weighting based on query variation reliability")
    
except Exception as e:
    print(f"QueryFusionRetriever error: {e}")
    print("Demonstrating Distribution-Based concept manually...")
    
    if not SCIPY_AVAILABLE:
        print("⚠️ Full statistical analysis requires scipy")
    
    # Manual demonstration with query variations derived from the main query
    query_variations = [
        DEMO_QUERIES["comprehensive"],  # Original query
        "machine learning approaches and methods",
        "different ML techniques and algorithms"
    ]
    
    print("Manual Distribution-Based Fusion with Query Variations:")
    all_results = {}
    variation_stats = []
    
    # Step 1: Collect results and analyze distributions
    for i, query_var in enumerate(query_variations):
        print(f"\nQuery variation {i+1}: {query_var}")
        nodes = base_retriever.retrieve(query_var)
        scores = [node.score or 0 for node in nodes]
        
        # Calculate distribution statistics
        mean_score = np.mean(scores) if scores else 0
        std_score = np.std(scores) if len(scores) > 1 else 1
        min_score = np.min(scores) if scores else 0
        max_score = np.max(scores) if scores else 1
        
        stats_info = {
            'mean': mean_score,
            'std': std_score,
            'min': min_score,
            'max': max_score,
            'nodes': nodes,
            'scores': scores
        }
        variation_stats.append(stats_info)
        
        print(f"Distribution stats: mean={mean_score:.3f}, std={std_score:.3f}")
        print(f"Score range: [{min_score:.3f}, {max_score:.3f}]")
        
        # Apply z-score normalization
        for node, score in zip(nodes, scores):
            node_id = node.node.node_id
            
            # Z-score normalization
            if std_score > 0:
                z_score = (score - mean_score) / std_score
            else:
                z_score = 0
            
            # Convert to [0,1] using sigmoid
            normalized_score = 1 / (1 + np.exp(-z_score))
            
            if node_id not in all_results:
                all_results[node_id] = {
                    'node': node,
                    'combined_score': 0,
                    'contributions': []
                }
            
            all_results[node_id]['combined_score'] += normalized_score
            all_results[node_id]['contributions'].append({
                'query': i,
                'original': score,
                'z_score': z_score,
                'normalized': normalized_score
            })
    
    # Step 2: Sort by combined distribution-based score
    sorted_results = sorted(
        all_results.values(),
        key=lambda x: x['combined_score'],
        reverse=True
    )
    
    print(f"\nCombined Distribution-Based Results (top 3):")
    for i, result in enumerate(sorted_results[:3], 1):
        print(f"{i}. Combined Score: {result['combined_score']:.4f}")
        print(f"   Statistical breakdown:")
        for contrib in result['contributions']:
            print(f"     Query {contrib['query']}: {contrib['original']:.3f} → "
                  f"z={contrib['z_score']:.2f} → {contrib['normalized']:.3f}")
        print(f"   Text: {result['node'].text[:100]}...")
        print()
    
    print("Distribution-Based Process:")
    print("1. Calculate mean and std for each query variation")
    print("2. Z-score normalize: z = (score - mean) / std")
    print("3. Sigmoid transform: normalized = 1 / (1 + exp(-z))")
    print("4. Sum normalized scores across variations")
    print("5. Results reflect statistical significance across all query forms")

# Show fusion mode comparison summary
print("\n" + "=" * 60)
print("FUSION MODES COMPARISON SUMMARY")
print("=" * 60)
print("All three modes tested with the same query for direct comparison:")
print(f"Query: {query}")
print()
print("Mode Characteristics:")
print("• RRF (reciprocal_rerank): Most robust, rank-based, scale-invariant")
print("• Relative Score: Preserves confidence, normalizes by max score")  
print("• Distribution-Based: Most sophisticated, statistical normalization")
print()
print("Choose based on your use case:")
print("- Production stability → RRF")
print("- Score interpretability → Relative Score")
print("- Statistical robustness → Distribution-Based")

6.3 DISTRIBUTION-BASED SCORE FUSION MODE DEMONSTRATION
Testing QueryFusionRetriever with dist_based_score mode:
This mode uses statistical analysis for the most sophisticated score fusion

Query: What are the main approaches to machine learning?
QueryFusionRetriever with dist_based_score will:
1. Generate query variations
2. Analyze score distributions for each variation
3. Apply statistical normalization (z-score, percentiles)
4. Combine with distribution-aware weighting

Distribution-Based Fusion Results:
1. Statistically Normalized Score: 0.7089
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Statistically Normalized Score: 0.6016
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....

3. Statistically Normalized Score: 0.5664
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...

Distribution-Based Benefits in

## Récupérateurs recommandés par cas d'utilisation

En vous basant sur les sources faisant autorité et les caractéristiques de chaque outil, voici les approches recommandées selon vos scénarios :

**Applications de FAQ / Questions-Réponses générales :**
- **Primaire** : Vector Index Retriever pour la compréhension sémantique.
- **Amélioration** : Combinaison avec le BM25 Retriever via le Query Fusion pour une approche hybride.
- **Avantage** : Allie la pertinence sémantique et la correspondance par mots-clés.
- **Source officielle** : "Pour les questions-réponses générales, utilisez un récupérateur d'index vectoriel, potentiellement combiné à un récupérateur BM25. Cette fusion combine pertinence sémantique et mots-clés."

**Documentation technique :**
- **Primaire** : BM25 Retriever pour la correspondance exacte des termes.
- **Amélioration** : Vector Index Retriever en soutien pour la flexibilité contextuelle.
- **Avantage** : Donne la priorité aux termes techniques précis tout en conservant une compréhension sémantique globale.
- **Source officielle** : "Pour les documents techniques, surtout quand les termes exacts sont prioritaires, considérez le BM25 comme récupérateur principal, l'index vectoriel servant de soutien contextuel."



**Documents longs :**
- **Primaire** : Auto Merging Retriever.
- **Avantage** : Ne récupère les versions "parents" (plus longues) que si suffisamment de fragments "enfants" sont trouvés, préservant ainsi le contexte.
- **Source officielle** : "Pour les documents longs, l'Auto Merging Retriever est une excellente option car il fusionne intelligemment les fragments pour conserver le contexte parent."

**Articles de recherche et publications académiques :**
- **Primaire** : Recursive Retriever.
- **Avantage** : Suit les citations et les références pour extraire du contenu pertinent dans les articles cités.
- **Source officielle** : "Pour les articles de recherche, utilisez le Recursive Retriever afin de récupérer le contenu pertinent provenant des travaux cités."

**Grandes collections de documents :**
- **Primaire** : Document Summary Index Retriever pour le filtrage initial.
- **Amélioration** : Suivi d'un Vector Index Retriever pour une recherche détaillée dans les documents filtrés.
- **Avantage** : Cible d'abord les documents pertinents avant d'effectuer une recherche granulaire.
- **Source officielle** : "Pour les ensembles de documents volumineux, utilisez l'index de résumés pour restreindre le nombre de documents, puis effectuez une recherche vectorielle dans ce sous-ensemble."

---
## ⏱️ À quoi s'attendre côté performances

Sur un CPU récent (i5/Ryzen 5) **sans GPU** :
- `llama3.2:1b` / `qwen3:1.7b` : réponses quasi instantanées, ~25–40 mots/s — idéal en classe.
- `llama3.2:3b` : ~15–25 mots/s, meilleures réponses.

Les étapes qui sollicitent le LLM (construction du *Document Summary Index*, génération de variantes dans *Query Fusion*) sont les plus lentes : prévoyez **1 à 3 minutes** la première fois sur une machine modeste. Les récupérateurs **BM25** et **Vector** restent, eux, immédiats.

**Conseils si une machine rame :**
- Passez `MODEL` à `"llama3.2:1b"` (une seule ligne à changer).
- Réduisez `Settings.context_window` (ex. `1024`) et `num_queries` dans les *Query Fusion*.
- Fermez les autres applications pour libérer de la RAM.
